In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim



# -------------------------------------------------
# 1. Device selection (ROCm or CPU)
# -------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# -------------------------------------------------
# 2. Load & preprocess data
# -------------------------------------------------
iris = load_iris()
X = iris.data
y = iris.target

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



In [ ]:

# Convert to tensors
X_train_gpu = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_gpu  = torch.tensor(X_test,  dtype=torch.float32).to(device)

y_train_gpu = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_gpu  = torch.tensor(y_test,  dtype=torch.long).to(device)

In [ ]:
epochs = 300
batch_size = 8
lr = 0.001

# -------------------------------------------------
# 3. Define a simple MLP model
# -------------------------------------------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 3)
        )
    def forward(self, x):
        return self.net(x)

model = MLP().to(device)

# -------------------------------------------------
# 4. Loss and optimizer
# -------------------------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# -------------------------------------------------
# 5. Training loop
# -------------------------------------------------


dataset = torch.utils.data.TensorDataset(X_train_gpu, y_train_gpu)
loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(loader):.4f}")

#


In [ ]:
# 6. Evaluation
# -------------------------------------------------
model.eval()
with torch.no_grad():
    preds = model(X_test_gpu)
    correct = (preds.argmax(dim=1) == y_test_gpu).sum().item()
    acc = correct / len(y_test)

print(f"Test accuracy: {acc:.4f}")